<a href="https://colab.research.google.com/github/oshriagronov/bistro-project/blob/main/exercise/ex10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1 - Installation**

In [4]:
# Install Java
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Download the latest Apache Spark version
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz

# Install findspark to connect Python with Spark
!pip install -q findspark


# **Step 2 - Environment Setup**

In [5]:
# Import the os module to interact with the operating system
import os
# Import findspark to locate the Spark installation
import findspark

# Set the environment variable for Java home directory (required for Spark to run)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# Set the environment variable for Spark home directory to the downloaded Spark path
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

# Initialize findspark to make pyspark importable within Python
findspark.init()


# **Step 3 - Create SparkSession**

In [6]:
# Import SparkSession class from PySpark SQL module
from pyspark.sql import SparkSession

# Create a SparkSession object, which is the entry point to use Spark functionality
  # Set the name of the Spark application to be "Big Data Example"
  # Create a new SparkSession or return an existing one
spark = SparkSession.builder.appName("Big Data Example").getOrCreate()

# **Real Dataset Example of Amazon Reviews**

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print(drive)
# df = spark.read.csv("", header=True, inferSchema=True)
# df.printSchema()
# df.show(50)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<module 'google.colab.drive' from '/usr/local/lib/python3.12/dist-packages/google/colab/drive.py'>


In [13]:
import os

# Define the path to your CSV file in Google Drive
# Note: 'My Drive' in the UI is mapped to 'MyDrive' in the file system
file_path = '/content/drive/MyDrive/Colab Notebooks/Copy-of-json-20251229-2107.json'

# Check if the file exists before trying to load it
if os.path.exists(file_path):
    # Load the CSV into a Spark DataFrame
    df_drive = spark.read.json(file_path)

    # Display the schema and first few rows
    df_drive.printSchema()
    df_drive.show(5)
else:
    print(f"File not found at: {file_path}. Please check the filename and folder path.")

root
 |-- created_at: string (nullable = true)
 |-- feed_id: long (nullable = true)
 |-- id: string (nullable = true)
 |-- value: string (nullable = true)

+--------------------+-------+--------------------+--------------------+
|          created_at|feed_id|                  id|               value|
+--------------------+-------+--------------------+--------------------+
|2025-12-06T08:10:...|3255183|0G1NXHRN33YRR5VH6...|{"temperature": 2...|
|2025-12-06T08:20:...|3255183|0G1NXQFQG7GVY0PWE...|{"temperature": 2...|
|2025-12-06T08:58:...|3255183|0G1NYDKP5A7K7JA72...|{"temperature": 2...|
|2025-12-06T09:00:...|3255183|0G1NYEPC9JTDXFFX2...|{"temperature": 2...|
|2025-12-06T09:01:...|3255183|0G1NYF092XBC2N11Y...|{"temperature": 2...|
+--------------------+-------+--------------------+--------------------+
only showing top 5 rows



# **Map Reduce Example**

In [16]:
import json

# Correct way to get an RDD from a DataFrame
rdd = df_drive.rdd

# Map step:
# 1. Access the 'value' column (which is a JSON string)
# 2. Parse the string into a dictionary
# 3. Extract temperature and map to (temp, 1)
def extract_temp(row):
    try:
        data = json.loads(row['value'])
        return (data.get('temperature'), 1)
    except:
        return (None, 0)

temp_counts = rdd.map(extract_temp).filter(lambda x: x[0] is not None)

# Reduce step: sum all counts per temperature value
result = temp_counts.reduceByKey(lambda a, b: a + b)

# Collect the results to the driver and print
print("Temperature counts:")
for temp, count in result.collect():
    print(f"Temp {temp} appears {count} times")

Temperature counts:
Temp 21.3 appears 58 times
Temp 22.6 appears 68 times
Temp 23.0 appears 39 times
Temp 23.3 appears 39 times
Temp 23.2 appears 47 times
Temp 23.4 appears 46 times
Temp 23.5 appears 45 times
Temp 23.6 appears 63 times
Temp 23.7 appears 16 times
Temp 23.8 appears 39 times
Temp 23.9 appears 35 times
Temp 24.0 appears 37 times
Temp 24.2 appears 60 times
Temp 23.1 appears 46 times
Temp 22.9 appears 38 times
Temp 22.7 appears 63 times
Temp 22.5 appears 103 times
Temp 22.4 appears 38 times
Temp 22.3 appears 68 times
Temp 22.2 appears 50 times
Temp 22.1 appears 58 times
Temp 22.0 appears 65 times
Temp 21.9 appears 95 times
Temp 22.8 appears 54 times
Temp 21.8 appears 22 times
Temp 21.7 appears 82 times
Temp 21.6 appears 31 times
Temp 21.5 appears 90 times
Temp 21.4 appears 61 times
Temp 24.1 appears 13 times
Temp 24.3 appears 9 times
Temp 24.4 appears 35 times
Temp 24.5 appears 20 times
Temp 24.6 appears 16 times
Temp 24.7 appears 19 times
Temp 24.8 appears 25 times
Temp 24.

soil counts:
Humidity 36.0 appears 149 times
Humidity 31.0 appears 58 times
Humidity 35.0 appears 80 times
Humidity 34.0 appears 76 times
Humidity 33.0 appears 76 times
Humidity 32.0 appears 93 times
Humidity 37.0 appears 216 times
Humidity 38.0 appears 181 times
Humidity 39.0 appears 141 times
Humidity 40.0 appears 154 times
Humidity 41.0 appears 205 times
Humidity 42.0 appears 297 times
Humidity 43.0 appears 262 times
Humidity 44.0 appears 236 times
Humidity 45.0 appears 214 times
Humidity 46.0 appears 141 times
Humidity 47.0 appears 80 times
Humidity 48.0 appears 78 times
Humidity 49.0 appears 126 times
Humidity 50.0 appears 116 times
Humidity 51.0 appears 129 times
Humidity 52.0 appears 122 times
Humidity 53.0 appears 52 times
Humidity 54.0 appears 29 times
Humidity 55.0 appears 13 times
Humidity 56.0 appears 8 times
Humidity 57.0 appears 4 times
Humidity 30.0 appears 15 times
Humidity 29.0 appears 10 times
Humidity 28.0 appears 15 times
Humidity 27.0 appears 1 times


In [18]:
# Map step:
# 1. Access the 'value' column (which is a JSON string)
# 2. Parse the string into a dictionary
# 3. Extract temperature and map to (temp, 1)
def extract_temp(row):
    try:
        data = json.loads(row['value'])
        return (data.get('soil'), 1)
    except:
        return (None, 0)

temp_counts = rdd.map(extract_temp).filter(lambda x: x[0] is not None)

# Reduce step: sum all counts per temperature value
result = temp_counts.reduceByKey(lambda a, b: a + b)

# Collect the results to the driver and print
print("soil counts:")
for temp, count in result.collect():
    print(f"soil {temp} appears {count} times")

soil counts:
soil 100 appears 3 times
soil 58 appears 44 times
soil 42 appears 50 times
soil 38 appears 84 times
soil 91 appears 3 times
soil 60 appears 46 times
soil 55 appears 52 times
soil 53 appears 80 times
soil 52 appears 49 times
soil 51 appears 54 times
soil 50 appears 50 times
soil 49 appears 50 times
soil 48 appears 50 times
soil 47 appears 57 times
soil 46 appears 45 times
soil 45 appears 51 times
soil 44 appears 67 times
soil 43 appears 65 times
soil 41 appears 71 times
soil 40 appears 73 times
soil 39 appears 79 times
soil 37 appears 102 times
soil 35 appears 81 times
soil 33 appears 84 times
soil 32 appears 62 times
soil 31 appears 68 times
soil 29 appears 58 times
soil 30 appears 52 times
soil 28 appears 48 times
soil 27 appears 57 times
soil 26 appears 66 times
soil 25 appears 60 times
soil 24 appears 88 times
soil 23 appears 84 times
soil 22 appears 61 times
soil 20 appears 63 times
soil 21 appears 75 times
soil 19 appears 32 times
soil 18 appears 22 times
soil 17 appe

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Function to get stats from RDD
def get_stats(rdd):
    data = rdd.map(lambda x: x).collect()
    if not data: return 0, 0
    return min(data), max(data)

# Calculate min/max for each
t_min, t_max = get_stats(rdd.map(lambda row: json.loads(row['value']).get('temperature')).filter(lambda x: x is not None))
h_min, h_max = get_stats(rdd.map(lambda row: json.loads(row['value']).get('humidity')).filter(lambda x: x is not None))
s_min, s_max = get_stats(rdd.map(lambda row: json.loads(row['value']).get('soil')).filter(lambda x: x is not None))

# Create subplots (one for each sensor)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Temperature", "Humidity", "Soil"),
    shared_yaxes=False
)

# Temperature Trace
fig.add_trace(go.Bar(name='Temp', x=['Min', 'Max'], y=[t_min, t_max], marker_color='indianred'), row=1, col=1)

# Humidity Trace
fig.add_trace(go.Bar(name='Humidity', x=['Min', 'Max'], y=[h_min, h_max], marker_color='lightsalmon'), row=1, col=2)

# Soil Trace
fig.add_trace(go.Bar(name='Soil', x=['Min', 'Max'], y=[s_min, s_max], marker_color='seagreen'), row=1, col=3)

fig.update_layout(title_text='Sensor Extremes (Min vs Max)', showlegend=False)
fig.show()